# Test OpenRouter Kimi K2.5 with LangChain

Quick validation of Kimi K2.5 via OpenRouter using LangChain's `ChatOpenAI`.

In [3]:
import os
from langchain_openai import ChatOpenAI

# Setup: uses OPENROUTER_API_KEY env var
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="moonshotai/kimi-k2.5",
    temperature=0,
)
print(f"Model: {llm.model_name}")
print(f"Base URL: {llm.openai_api_base}")

Model: moonshotai/kimi-k2.5
Base URL: https://openrouter.ai/api/v1


## 1. Simple Invoke Test

In [4]:
response = llm.invoke("What is spatial transcriptomics in one sentence?")
print(response.content)
print(f"\nUsage: {response.usage_metadata}")

 Spatial transcriptomics is a technique that maps gene expression data to their specific physical locations within intact tissue samples, preserving the spatial organization and context of molecular information.

Usage: {'input_tokens': 20, 'output_tokens': 405, 'total_tokens': 425, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 561}}


## 2. Streaming Test

In [5]:
for chunk in llm.stream("Explain ligand-receptor interactions in 2 sentences."):
    print(chunk.content, end="", flush=True)
print()  # newline

Ligand-receptor interactions occur when specific signaling molecules (ligands) bind to complementary protein receptors on or within cells with high specificity and affinity, acting as molecular recognition events. This binding induces conformational changes in the receptor that trigger signal transduction pathways, converting extracellular chemical messages into intracellular responses such as gene expression changes or metabolic shifts.


## 3. Tool Calling Test

In [6]:
from langchain_core.tools import tool


@tool
def search_pubmed(query: str, max_results: int = 5) -> str:
    """Search PubMed for biomedical literature."""
    return f"Found {max_results} results for: {query}"


llm_with_tools = llm.bind_tools([search_pubmed])
response = llm_with_tools.invoke(
    "Search PubMed for recent papers on MERFISH spatial transcriptomics"
)

print("Content:", response.content)
print("Tool calls:", response.tool_calls)

Content: 
Tool calls: [{'name': 'search_pubmed', 'args': {'query': 'MERFISH spatial transcriptomics', 'max_results': 10}, 'id': 'functions.search_pubmed:0', 'type': 'tool_call'}]


## 4. Multi-turn Conversation Test

In [7]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a bioinformatics expert. Be concise."),
    HumanMessage(content="What is LIANA used for?"),
]

response = llm.invoke(messages)
print(response.content)

**LIANA** (Ligand-receptor ANalysis frAmework) is a computational tool for **inferring cell-cell communication** from single-cell RNA sequencing (scRNA-seq) data.

**Primary use:** It predicts and ranks ligand-receptor interactions between cell types to map intercellular signaling networks.

**Key features:**
- **Integrative:** Aggregates predictions from multiple established methods (CellPhoneDB, CellChat, NATMI, SingleCellSignalR, etc.)
- **Consensus scoring:** Ranks interactions by agreement across methods to reduce false positives
- **Resource aggregation:** Combines multiple ligand-receptor databases (CellPhoneDB, CellChatDB, connectomeDB2020, etc.)
- **Downstream analysis:** Supports differential cell-cell communication between conditions and visualization

**Input:** Single-cell expression matrices + cell type annotations  
**Output:** Ranked lists of putative ligand-receptor pairs between cell populations, with consensus scores and statistical significance.

It's available as a